# Statistics for Pro-VET Examples

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [2]:
methods_summary_values = pd.read_csv("./results/methods_summary_info.csv")
methods_summary_values



,Unnamed: 0,EZpep_Sonic_N1,EZpep_Sonic_N2,EZpep_Sonic_N3,EZpep_Sonic_N4,EZpep_Sonic_N5,SDC_Amide_Sonic_N1,SDC_Amide_Sonic_N2,SDC_Amide_Sonic_N3,SDC_Amide_Sonic_N4,...,Strap_Sonic_N1,Strap_Sonic_N2,Strap_Sonic_N3,Strap_Sonic_N4,Strap_Sonic_N5,Urea_Sonic_N1,Urea_Sonic_N2,Urea_Sonic_N3,Urea_Sonic_N4,Urea_Sonic_N5
0,Protein Groups,5417,5536,5371,5364,5400,5980,6057,5972,6079,...,3946,4205,4289,4225,4133,5332,5283,5155,5170,5069
1,Peptides,30877,30773,29688,30462,30094,33741,32783,33807,33789,...,17534,19925,19703,18731,19057,33624,33256,31999,32123,32128
2,PSM,36667,36944,35441,36481,36063,39333,38213,39615,39268,...,20084,22942,22486,21372,22041,41490,41350,40264,40256,39329


In [18]:
all_groups = methods_summary_values.T
# 1. Set the first row as the column names
all_groups.columns = all_groups.iloc[0]

# 2. Drop the first row and reset the index
all_groups = all_groups[1:].reset_index(drop=True)
all_groups = all_groups.astype(int)

In [19]:
all_groups['group'] = ['A', 'A', 'A', 'A', 'A',
                             'B', 'B', 'B', 'B', 'B',
                             'C', 'C', 'C', 'C', 'C',
                             'D', 'D', 'D', 'D', 'D',
                             'E', 'E', 'E', 'E', 'E']

protein_groups = all_groups.iloc[: , 0]
peptides = all_groups.iloc[: , 1]
PSMs = all_groups.iloc[: , 2]
all_groups

Unnamed: 0,Protein Groups,Peptides,PSM,group
0,5417,30877,36667,A
1,5536,30773,36944,A
2,5371,29688,35441,A
3,5364,30462,36481,A
4,5400,30094,36063,A
5,5980,33741,39333,B
6,6057,32783,38213,B
7,5972,33807,39615,B
8,6079,33789,39268,B
9,5933,31568,36893,B


# Protein Groups

In [20]:

protein_group_A = protein_groups.iloc[0:5]
protein_group_B = protein_groups.iloc[5:10]
protein_group_C = protein_groups.iloc[10:15]
protein_group_D = protein_groups.iloc[15:20]
protein_group_E = protein_groups.iloc[20:]



In [21]:
# Perform one-way ANOVA
f_stat, p_value = stats.f_oneway(protein_group_A, protein_group_B, protein_group_C, protein_group_D, protein_group_E)

print("F-statistic:", f_stat)
print("P-value:", p_value)

F-statistic: 277.8585817495875
P-value: 3.2237656634294974e-17


# Perform Tukey's Honestly Significant Difference (HSD) post hoc test

In [22]:
all_groups

Unnamed: 0,Protein Groups,Peptides,PSM,group
0,5417,30877,36667,A
1,5536,30773,36944,A
2,5371,29688,35441,A
3,5364,30462,36481,A
4,5400,30094,36063,A
5,5980,33741,39333,B
6,6057,32783,38213,B
7,5972,33807,39615,B
8,6079,33789,39268,B
9,5933,31568,36893,B


In [24]:
# Perform Tukey's HSD
tukey = pairwise_tukeyhsd(endog = all_groups['Protein Groups'], groups = all_groups['group'], alpha=0.05)

# View the results
print(tukey)

   Multiple Comparison of Means - Tukey HSD, FWER=0.05    
group1 group2 meandiff p-adj    lower      upper    reject
----------------------------------------------------------
     A      B    586.6    0.0   413.0705   760.1295   True
     A      C    118.0 0.2861   -55.5295   291.5295  False
     A      D  -1258.0    0.0 -1431.5295 -1084.4705   True
     A      E   -215.8 0.0105  -389.3295   -42.2705   True
     B      C   -468.6    0.0  -642.1295  -295.0705   True
     B      D  -1844.6    0.0 -2018.1295 -1671.0705   True
     B      E   -802.4    0.0  -975.9295  -628.8705   True
     C      D  -1376.0    0.0 -1549.5295 -1202.4705   True
     C      E   -333.8 0.0001  -507.3295  -160.2705   True
     D      E   1042.2    0.0   868.6705  1215.7295   True
----------------------------------------------------------


### Effect Size

In [31]:
import pandas as pd
import pingouin as pg


# Run ANOVA to automatically get eta-squared (eta2)
anova_table = pg.anova(data = all_groups, dv='Protein Groups', between='group')
print(anova_table[['Source', 'F', 'p_unc', 'np2']])  # np2 is partial eta-squared


  Source           F         p_unc       np2
0  group  277.858582  3.223766e-17  0.982323


In [29]:
anova_table


,Source,ddof1,ddof2,F,p_unc,np2
0,group,2,9,216.8,2.438035e-08,0.979666


In [ ]:
How to interpret np2 = 0.9823

This is an extremely large effect size.

Typical benchmarks:

np²	Interpretation
0.01	small
0.06	medium
0.14	large
>0.50	very large

👉 Your result:

0.98 = near-total separation between groups
Group differences dominate the dataset